# CipherMark : diagnostics C2 et C3 sur GPU

Deux mesures demandées par le rapport d'examen, trop lentes sur CPU :

1. **C2, recadrage** : recherche aveugle de la position d'un morceau recadré à 0,9 (100 images, grille au pas de 2 px), sur images tatouées ET non tatouées (faux positifs).
2. **C3, marge souple du hash** : ne comparer que les bits fiables du hash, sur 1 000 images (environ un million de paires étrangères).

**Exécution → Tout exécuter.** Il faut un GPU : Exécution → Modifier le type d'exécution → T4.
Les résultats sont copiés sur le Drive, dans `ciphermark/13-colab-diagnostics-c2-c3/resultats/`.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Execution -> Modifier le type d'execution -> T4, puis relancer.")
print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, subprocess

RACINE_DRIVE = "/content/drive/MyDrive/ciphermark"
DOSSIER      = f"{RACINE_DRIVE}/13-colab-diagnostics-c2-c3"
DEPOT        = "/content/code-memoire"
SORTIE       = f"{DOSSIER}/resultats"
os.makedirs(SORTIE, exist_ok=True)

# 1) le code : archive du Drive, extraite a cote puis deplacee (l'archive a
#    un dossier racine, cf. carnet de campagne)
if not os.path.isdir(os.path.join(DEPOT, "distseal", "utils")):
    tmp = "/content/_extraction"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    subprocess.run(["tar", "xzf", f"{RACINE_DRIVE}/00-code-source-a-jour.tar.gz", "-C", tmp], check=True)
    racine = next(r for r, _, _ in os.walk(tmp) if os.path.isdir(os.path.join(r, "distseal", "utils")))
    shutil.rmtree(DEPOT, ignore_errors=True); shutil.move(racine, DEPOT)

# 2) les deux scripts de diagnostic, plus recents que l'archive
for s in ("diag_resync_recadrage.py", "diag_marge_hash.py"):
    shutil.copy(f"{DOSSIER}/{s}", f"{DEPOT}/scripts/ciphermark/{s}")

# 3) le tatoueur de reference 64 bits (meme MD5 que le local : b4dbbfdf2302...)
os.makedirs(f"{DEPOT}/runs", exist_ok=True)
ck = f"{DEPOT}/runs/ciphermark_64bits_checkpoint.pth"
if not os.path.exists(ck):
    shutil.copy(f"{RACINE_DRIVE}/01-tatoueur-modeles-de-reference-64-et-128-bits/"
                "phaseA2_64bits_stable_checkpoint.pth", ck)

# 4) le corpus d'evaluation : COCO val2017, jamais vu a l'entrainement
if not os.path.isdir("/content/corpus-coco-5000/toutes"):
    subprocess.run(["tar", "xzf", f"{DOSSIER}/corpus-coco-5000.tar.gz", "-C", "/content"], check=True)
print("images :", len(os.listdir("/content/corpus-coco-5000/toutes")))
os.chdir(DEPOT)
print("pret :", DEPOT)

In [ ]:
import importlib.util
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "torchmetrics",
                "einops", "omegaconf", "safetensors"])
for mod, paquet in (("reedsolo", "reedsolo"), ("Crypto", "pycryptodome"),
                    ("huggingface_hub", "huggingface_hub")):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet])
print("dependances installees")

## C2 : recherche aveugle du recadrage (environ 10 à 20 min)

In [ ]:
env = dict(os.environ, PYTHONPATH=f"{DEPOT}/deps:{DEPOT}", DEPOT=DEPOT,
           CORPUS="/content/corpus-coco-5000/toutes", GRILLE="0.9", PAS="2", N="100")
journal = f"{DEPOT}/runs/diag_resync_recadrage_N100_grille0.9_pas2.log"
with open(journal, "w") as f:
    p = subprocess.Popen([sys.executable, "scripts/ciphermark/diag_resync_recadrage.py"],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ligne in p.stdout:
        f.write(ligne)
        if "warn" not in ligne.lower():
            print(ligne, end="")
code = p.wait()
print(f"\ncode de sortie : {code}" + ("  (-9 : tue, souvent par manque de RAM)" if code == -9 else ""))
for fic in (journal, f"{DEPOT}/runs/diag_resync_recadrage_N100_grille0.9_pas2.json"):
    if os.path.exists(fic):
        shutil.copy(fic, SORTIE)
print("copie sur le Drive :", SORTIE)

## C3 : marge souple du hash, 1 000 images (environ 10 à 20 min)

In [ ]:
env = dict(os.environ, PYTHONPATH=f"{DEPOT}/deps:{DEPOT}")
journal = f"{DEPOT}/runs/diag_marge_hash_N1000.log"
with open(journal, "w") as f:
    p = subprocess.Popen([sys.executable, "scripts/ciphermark/diag_marge_hash.py",
                          "--n-images", "1000", "--batch", "32", "--threads", "2",
                          "--corpus", "/content/corpus-coco-5000"],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ligne in p.stdout:
        f.write(ligne)
        if "warn" not in ligne.lower():
            print(ligne, end="")
code = p.wait()
print(f"\ncode de sortie : {code}" + ("  (-9 : tue, souvent par manque de RAM)" if code == -9 else ""))
for fic in (journal, f"{DEPOT}/runs/diag_marge_hash_N1000.json"):
    if os.path.exists(fic):
        shutil.copy(fic, SORTIE)
print("copie sur le Drive :", SORTIE)